In [1]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, dotenv, glob, gc, zipfile, subprocess
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from rasterio.features import rasterize, shapes
from rasterio.merge import merge
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions, functions
from shapely.geometry import Polygon
from shapely import force_2d
from pyflwdir import dem
from tqdm import tqdm
from owslib.wcs import WebCoverageService
from whitebox.whitebox_tools import WhiteboxTools
from terracatalogueclient import Catalogue
dotenv.load_dotenv()
wtb = WhiteboxTools()
wtb.set_verbose_mode(False)
np.random.seed(42)

## Child Functions

In [8]:
def create_forcing(time, ny, nx, values, single_value=True):
    if single_value:
        nt = len(time)
        data = np.zeros((nt, ny, nx), dtype=np.float32)
        for i in range(nt):
            data[i, :, :] = values[i]
    

    return data


In [2]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
lake_path = os.path.join(sample_folder, 'BrusdalLake.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
NODATA_FLOAT, NODATA_INT = -9999.0, 0
raw_dir = os.path.normpath(os.path.join(f'{test_folder}/data/raw'))
if not os.path.exists(raw_dir): os.makedirs(raw_dir)
raw_path = os.path.normpath(os.path.join(raw_dir, "dtm_raw.tif"))
lake = gpd.read_file(lake_path).to_crs(terrain.rio.crs)
land_dir = os.path.normpath(os.path.join(test_folder, 'data/landcover'))
if not os.path.exists(land_dir): os.makedirs(land_dir)

## Create a raw terrain that is clipped to catchment

In [3]:
# Clip terrain to catchment
catchment_UTM = catchment.to_crs(terrain.rio.crs)
temp = catchment_UTM.copy()
temp['geometry'] = temp['geometry'].buffer(10)
terrain_clipped = flow_functions.clip_catchment(temp, terrain)
terrain_clipped.rio.to_raster(raw_path)

## Process river data

In [4]:
# Create river from DEM if it doesn't exist
if not os.path.exists(river_path):
    threshold, min_length = 0.1, 100
    with rasterio.open(raw_path) as src:
        dem_array = src.read(1).astype(np.float32)
        transform, profile = src.transform, src.profile
        crs, nodata = src.crs, src.nodata
    mask = np.isnan(dem_array) | (dem_array == nodata)
    filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
    flwdir_array = np.where(mask, NODATA_INT, flwdir_array)
    flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
    uparea = flw.upstream_area(unit="km2")
    river_mask = uparea > threshold
    features = flw.streams(river_mask)
    gdf = gpd.GeoDataFrame.from_features(features, crs=crs)
    lake['geometry'] = lake['geometry'].buffer(10)
    # Clipp river to lake
    clipped_river = gdf.overlay(lake, how='difference')
    clipped_river['lenght'] = clipped_river['geometry'].length
    clipped_river = clipped_river[clipped_river['lenght'] > min_length]
    clipped_river.reset_index(drop=True, inplace=True)
    clipped_river = clipped_river[['geometry']]
    river = clipped_river.reindex(columns=['rivwth', 'rivdph', 'geometry'])
else:
    river = gpd.read_file(river_path)
    river = river.rename(columns={'width': 'rivwth', 'depth': 'rivdph'})
    river = river[['rivwth', 'rivdph', 'geometry']]
# Create a random value for each river
cols = {'rivwth': (0.05, 2), 'rivdph': (1, 5)}
river_cols = river.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river[col] = pd.to_numeric(river[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river[col].isna() | (river[col] == 'None')
    river.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river = river[river.is_valid].reset_index(drop=True)
# Process river
river["geometry"] = river.geometry.apply(lambda g: force_2d(g))
# Write file
river.to_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')), driver='GPKG')

## Create template hydro data

In [22]:
# Prepare template raster dataset
with rasterio.open(raw_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile, transform, crs = src.profile, src.transform, src.crs
hydro_dir = os.path.normpath(f'{test_folder}/data/hydro')
if os.path.exists(hydro_dir): shutil.rmtree(hydro_dir)
os.makedirs(hydro_dir)
NODATA_DEM, NODATA_INT = -9999.0, 0
with rasterio.open(raw_path) as src:
    dem_array = src.read(1).astype(np.float32)
    transform, profile = src.transform, src.profile
    crs, nodata = src.crs, src.nodata
# Fill depressions
filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
flwdir_array = np.where(filled_array == NODATA_DEM, NODATA_INT, flwdir_array)
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
gradient_array = filled_array.copy()
gradient_array[gradient_array == NODATA_DEM] = 0
gy, gx = np.gradient(gradient_array, dy, dx)
slope_array = np.sqrt(gx**2 + gy**2)
slope_array[filled_array == NODATA_DEM] = NODATA_DEM
# Create basins
flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
basins_array = flw.basins()
basins_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, basins_array)
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
uparea_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, uparea_array)
# Create stream mask and stream order
stream_mask = uparea_array > 0
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
strord_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, strord_array)
# Create upstream grid
upgrid_array = flw.upstream_area(unit='cell')
upgrid_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, upgrid_array)
# Create river width
river = gpd.read_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')))
shape = ((geom, value) for geom, value in zip(river.geometry, river["rivwth"]))
rivwth_array = rasterize(
    shapes=shape, out_shape=(src.height, src.width),
    transform=transform, fill=NODATA_DEM, dtype="float32"
)
rivwth_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, rivwth_array)
profile_float = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
profile_int = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
flow_functions.write_geotiff(filled_array, profile_float, os.path.join(hydro_dir, 'elevtn.tif'))
flow_functions.write_geotiff(flwdir_array, profile_int, os.path.join(hydro_dir, 'flwdir.tif'))
flow_functions.write_geotiff(slope_array, profile_float, os.path.join(hydro_dir, 'lndslp.tif'))
flow_functions.write_geotiff(basins_array, profile_float, os.path.join(hydro_dir, 'basins.tif'))
flow_functions.write_geotiff(uparea_array, profile_float, os.path.join(hydro_dir, 'uparea.tif'))
flow_functions.write_geotiff(strord_array, profile_float, os.path.join(hydro_dir, 'strord.tif'))
flow_functions.write_geotiff(upgrid_array, profile_float, os.path.join(hydro_dir, 'upgrid.tif'))
flow_functions.write_geotiff(rivwth_array, profile_float, os.path.join(hydro_dir, 'rivwth.tif'))

## Prepare forcing data from the customized area

In [6]:
# Read weather data
weather_path = os.path.join(sample_folder, 'weather_2025.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']

In [9]:
# Create forcing nc file
time, crs = weather_new.index.to_numpy(), terrain.rio.crs
if crs is None: raise ValueError("Terrain has no crs")
ny, nx = terrain.rio.height, terrain.rio.width
forcing = {
    'precip': ['precip_mm', 'mm/h'], 'temp': ['temp_C', 'degC'],
    'kin': ['shortwave_Wm2', 'W/m^2'], 'kout': ['longwave_Wm2', 'W/m^2'],
    'wind': ['wind_mps', 'm/s'], 'press_msl': ['pressure', 'Pa']
}
forcing_dir = os.path.join(test_folder, 'data/forcing')
os.makedirs(forcing_dir, exist_ok=True)
out_path, datasets = os.path.join(forcing_dir, "my_forcing.nc"), {}
for var, (col, unit) in forcing.items():
    data = weather_new[col].values.astype(np.float32)
    if var == "precip": data = data * 24.0
    data_3d = create_forcing(time, ny, nx, data)
    data_3d = np.nan_to_num(data_3d, nan=0.0, posinf=0.0, neginf=0.0)
    datasets[var] = (('time', 'y', 'x'), data_3d, {'units': unit})
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": terrain.y.values, "x": terrain.x.values}
)
ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
ds_final.rio.write_crs(crs, inplace=True)
ds_final["time"].encoding = {
    "units": "hours since 1900-01-01 00:00:00",
    "calendar": "proleptic_gregorian"
}
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process soil data

In [ ]:
# Initialize variables
soil_types = {
    'clay': 'clyppt', 'sand': 'sndppt', 'silt': 'sltppt', 
    'bdod': 'bd', 'soc': 'oc', 'phh2o': 'ph'
}
depths = {
    '0-5cm_mean': 'sl1', '5-15cm_mean': 'sl2', '15-30cm_mean': 'sl3',
    '30-60cm_mean': 'sl4', '60-100cm_mean': 'sl5', '100-200cm_mean': 'sl6'
}
soil_dir = os.path.join(f'{test_folder}/data/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
min_lon, min_lat, max_lon, max_lat = catchment.total_bounds
bbox = (float(min_lon), float(min_lat), float(max_lon), float(max_lat))

In [ ]:
# Create soil thickness
with rasterio.open(terrain_path) as src:
    meta = src.meta.copy()
meta.update({"dtype": "float32", "nodata": -9999.0})
data = np.ones((meta["height"], meta["width"]), dtype="float32") * 100
data[data == meta["nodata"]] = 100
with rasterio.open(os.path.join(soil_dir, 'soilthickness.tif'), "w", **meta) as dst:
    dst.write(data, 1)

In [ ]:
# Download soil data from ISRIC: https://files.isric.org/soilgrids/latest/data/
for item, name in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading soil data'):
    wcs = WebCoverageService(f'https://maps.isric.org/mapserv?map=/map/{item}.map', version='1.0.0')
    for type, value in depths.items():
        idx = f'{item}_{type}'
        response = wcs.getCoverage(
            identifier=idx, crs='EPSG:4326', bbox=bbox,
            format='image/tiff', resx=0.0025, resy=0.0025
        )
        with MemoryFile(response.read()) as memfile:
            with memfile.open() as src:
                data = rioxarray.open_rasterio(src, masked=True)
                data_reprojected = data.rio.reproject_match(terrain)
            data_reprojected.rio.to_raster(os.path.join(soil_dir, f'{name}_{value}.tif'))

## Process land cover

In [8]:
# Get landcover data from ESA worldcover
user_name, password = os.getenv('ESA_USERNAME'), os.getenv('ESA_PASSWORD')
catalogue = Catalogue().authenticate_non_interactive(user_name, password)
area = catchment.copy()
if area.crs != 'EPSG:4326': area = area.to_crs('EPSG:4326')
minx, miny, maxx, maxy = area.total_bounds
bbox = Polygon.from_bounds(minx, miny, maxx, maxy)
download_dir = os.path.join(land_dir, 'downloads')
if os.path.exists(download_dir): shutil.rmtree(download_dir)
# # Get name of landcover layer
# collections = catalogue.get_collections()
layers = [
    # 'urn:eop:VITO:ESA_WorldCover_10m_2020_V1', 
    'urn:eop:VITO:ESA_WorldCover_10m_2021_V2'
]
# Search for products in the WorldCover collection
product = catalogue.get_products(layers, geometry=bbox)
catalogue.download_products(product, download_dir, force=True)
pattern = os.path.join(download_dir, "**", "*_Map.tif")
files = glob.glob(pattern, recursive=True)
if len(files) == 0: raise ValueError("No *_Map.tif files found in directory")
# print(f"Found {len(files)} tiles")
if len(files) == 1:
    src_files = [rasterio.open(files[0])]
    mosaic, transform = src_files[0].read(), src_files[0].transform
elif len(files) > 1:
    src_files = [rasterio.open(f) for f in files]
    # Merge (mosaic)
    mosaic, transform = merge(src_files)
# Copy metadata
out_meta = src_files[0].meta.copy()
out_meta.update({
    "height": mosaic.shape[1], "width": mosaic.shape[2],
    "transform": transform, "compress": "lzw"
})
merge_path = os.path.join(land_dir, 'merged.tif')
with rasterio.open(merge_path, "w", **out_meta) as dest:
    dest.write(mosaic)
# Close files
for src in src_files: src.close()
shutil.rmtree(download_dir)
# Clip raster to catchment
ds = rioxarray.open_rasterio(merge_path).squeeze()
land_temp = ds.squeeze()
# area['geometry'] = area['geometry'].buffer(0.001)
land_clip = flow_functions.clip_catchment(area, land_temp, land_temp.rio.nodata)
path = os.path.join(land_dir, 'esa_worldcover.tif')
land_clip = land_clip.rio.reproject(terrain.rio.crs)
land_clip.rio.to_raster(path)
del land_temp, land_clip, ds
gc.collect()
functions.safe_remove(merge_path)
# Save look up table
csv_path = r"backend\src\flow_samples\landcover\esa_worldcover_mapping.csv"
df = pd.read_csv(csv_path)
df.to_csv(os.path.join(land_dir, 'esa_worldcover.csv'), index=False)

C:\Users\vanln\AppData\Local\Temp\ipykernel_22736\2895255780.py:45: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area['geometry'] = area['geometry'].buffer(0.001)


In [25]:
# Land cover data from CORINE 2018
# Get raster information
with rasterio.open(terrain_path) as src:
    profile = src.profile
# Convert the original data from raster to vector
corine_path = r"backend\src\flow_samples\landcover\U2018_CLC2018_V2020_20u1.zip"
csv_path = r"backend\src\flow_samples\landcover\corine_mapping.csv"
area = catchment.copy()
with zipfile.ZipFile(corine_path, "r") as zip_ref:
    name = os.path.basename(corine_path).replace(".zip", ".tif")
    with zip_ref.open(name) as f:
        data = f.read()
        with MemoryFile(data) as memfile:
            with rioxarray.open_rasterio(memfile) as src:
                corine_land = src.squeeze()
                area = area.to_crs(corine_land.rio.crs)
                # area['geometry'] = area['geometry'].buffer(10)
                land_clip = flow_functions.clip_catchment(area, corine_land, corine_land.rio.nodata)
land_clip = land_clip.rio.reproject(terrain.rio.crs)
data, nodata = land_clip.values.astype(np.int32), land_clip.rio.nodata
lookup = np.full(256, 999, dtype=np.int32)
for k, v in flow_functions.corine_codes.items():
    lookup[k] = v[0]
data_new = lookup[data]
mask = (data_new == nodata)
data_new[mask], profile_corine = 999, profile.copy()
profile_corine.update({'dtype': np.int32, 'nodata': 999, 'count': 1, 'compress': 'lzw'})
path = os.path.join(test_folder, 'data/landcover', 'corine.tif')
flow_functions.write_geotiff(data_new, profile_corine, path)
# Save look up table
df = pd.read_csv(csv_path)
df['canopy_gap_fraction'] = df['corine'].astype(int).map(lambda x: flow_functions.canopy_gap_fraction.get(x, 999.0))
df.to_csv(os.path.join(land_dir, 'corine.csv'), index=False)

In [18]:
np.unique(land_clip.values.astype(np.int32))

array([-128,    2,    3,   10,   21,   23,   24,   27,   32,   41],
      dtype=int32)

In [50]:
np.unique(data_new)

array([112, 121, 141, 243, 311, 312, 322, 333, 512, 999], dtype=int32)

## Run HydroMT

In [26]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-31 22:59:45,499 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-31 22:59:45,657 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-31 22:59:45,739 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-31 22:59:45,739 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-31 22:59:45,771 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-31 22:59:45,771 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-31 22:59:45,774 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-31 22:59:45,775 - hydromt.model.model - model - INFO - build: setup_config
2026-05-31 22:59:45,776 - hydromt.m

## Run simulation

In [38]:
wflow_exe = r"D:\Programming_Codes\Hydro-AI-Platform\backend\softs\wflow 1.0.2\wflow_cli\bin\wflow_cli.exe"
cwd = r"D:\Programming_Codes\Hydro-AI-Platform\test\model"
toml_path = r"D:\Programming_Codes\Hydro-AI-Platform\test\model\wflow_sbm.toml"

In [ ]:
# Add more variables to the config file
# 1. Add "vegetation_canopy__gap_fraction_gap_fraction" to config.yml
# 2. Add "soil_water__vertical_saturated_hydraulic_conductivity_scale_parameter" to config.yml
# 3. Add "soil_layer_water__brooks_corey_exponent" to config.yml


In [40]:
cmd = [wflow_exe, toml_path]
process = subprocess.Popen(
    cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='')
process.wait()
print('Return code:', process.returncode)

[ Info: Wflow version v1.0.2
[ Info: Initialize model variables for model type sbm.
â”Œ Info: Cyclic parameters are provided by
â”” D:\Programming_Codes\Hydro-AI-Platform\test\model\staticmaps.nc.
â”Œ Info: Forcing parameters are provided by
â”” D:\Programming_Codes\Hydro-AI-Platform\test\model\inmaps_forcing.nc.
â”Œ Info: Set atmosphere_water__precipitation_volume_flux using netCDF variable
â”” precip as forcing parameter.
â”Œ Info: Set atmosphere_air__temperature using netCDF variable temp as forcing
â”” parameter.
â”Œ Info: Set land_surface_water__potential_evaporation_volume_flux using
â”” netCDF variable pet as forcing parameter.
â”Œ Info: General model settings
â”‚   snow = true
â”‚   gravitational_snow_transport = true
â”‚   glacier = false
â”‚   reservoirs = false
â”‚   pits = false
â””   water_demand = false
[ Info: Set subbasin_location__count using netCDF variable subcatchment.
â”Œ Info: Set basin__local_drain_direction using netCDF variable
â”” local_drain_direction.
[ Info

In [ ]:
with rasterio.open(os.path.join(model_path, 'maps', 'elevtn.tif')) as src:
    data = src.read(1)
    print("elevtn.tif - nodata count:", np.isnan(data).sum())

In [42]:
with xr.open_dataset(r"test\model\staticmaps.nc") as ds:
    for item in ds.data_vars:
        print(f"{item}")

meta_subgrid_outlet_x
meta_subgrid_outlet_y
local_drain_direction
subcatchment
meta_upstream_area
meta_subgrid_area
meta_subgrid_elevation
meta_streamorder
meta_subgrid_outlet_idx
land_elevation
land_slope
river_mask
river_length
river_slope
river_width
river_depth
river_manning_n
soil_theta_s
soil_theta_r
soil_thickness
soil_brooks_corey_c
soil_ksat_vertical
meta_soilgrids_2020_ksat_vertical_2.5cm
meta_soilgrids_2020_ksat_vertical_10.0cm
meta_soilgrids_2020_ksat_vertical_22.5cm
meta_soilgrids_2020_ksat_vertical_45.0cm
meta_soilgrids_2020_ksat_vertical_80.0cm
meta_soilgrids_2020_ksat_vertical_150.0cm
soil_f_
soil_f
meta_soil_texture
meta_landuse
vegetation_kext
land_manning_n
soil_compacted_fraction
vegetation_root_depth
vegetation_leaf_storage
vegetation_wood_storage
land_water_fraction
vegetation_crop_factor
vegetation_feddes_alpha_h1
vegetation_feddes_h1
vegetation_feddes_h2
vegetation_feddes_h3_high
vegetation_feddes_h3_low
vegetation_feddes_h4


In [32]:
with xr.open_dataset(r"test\model\staticmaps.nc") as ds:
    print(ds['vegetation_kext'].values, ds['vegetation_kext'].values.shape)

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]] (44, 131)
